<font size = "4">

For some of these questions, you are either asked to explicitly add methods to the `Graph` class, or adding the methods to the Graph class would be advantageous to solve the problem.

Here is an example of how you can add a method to the `Graph` class:

In [1]:
from adjacency_graph import Graph

# write function
def check_for_key(self, key):
    if key in self._vertices:
        print("Yep, it's in here!")
    else:
        print(f"Nope, no {key} here!")

# Assign function to a new method of the class
Graph.check_for_key = check_for_key

G = Graph()
G.add_edge("A", "B")

G.check_for_key("A")
G.check_for_key("X")

Yep, it's in here!
Nope, no X here!


<font size = "3">

**(Q1)** Write a function or method that performs a depth-first search (DFS) and a topological sort at the same time. That is, instead of doing DFS and then sorting the vertices in a post-processing step, build up a data structure containing the keys/vertices that produces the topologically sorted order.

In [2]:
from adjacency_graph import Graph

def dfs_visit_topo(self, start, topo_order):
    start.color = "gray"
    self._time += 1
    start.discovery_time = self._time
    for next_vertex in start.get_neighbors():
        if next_vertex.color == "white":
            next_vertex.set_previous(start)
            self.dfs_visit_topo(next_vertex, topo_order)
    start.color = "black"
    self._time += 1
    start.closing_time = self._time
    topo_order.insert(0, start.key)  # prepend on finish = topological order

def dfs_topo(self):
    topo_order = []
    for vertex in self:
        if vertex.color == "white":
            self.dfs_visit_topo(vertex, topo_order)
    return topo_order

Graph.dfs_visit_topo = dfs_visit_topo
Graph.dfs_topo = dfs_topo

# test 
G = Graph()
G.add_edge("A", "B")
G.add_edge("A", "C")
G.add_edge("B", "D")
G.add_edge("C", "D")
G.add_edge("D", "E")

print(G.dfs_topo())  # should be a valid topological ordering

['A', 'C', 'B', 'D', 'E']


<font size = "3">

**(Q2)** Write the `transpose` method for the `Graph` class. If `G` is a directed graph, then `G.transpose()` will return its transpose.

In [3]:
from adjacency_graph import Graph

def transpose(self):
    gt = Graph()
    for (from_key, to_key), weight in self._edges.items():
        gt.add_edge(to_key, from_key, weight)
    return gt

Graph.transpose = transpose

# test
G = Graph()
G.add_edge("A", "B")
G.add_edge("B", "C")
G.add_edge("C", "A")

GT = G.transpose()
print(list(GT.get_edges()))  # edges should be reversed

[('B', 'A'), ('C', 'B'), ('A', 'C')]


<font size = "3">

**(Q3)** Create a modification of the depth-first search (DFS) method that returns a new graph representing the strongly connected components of the graph. (See the first two images of "lecture_24_b_scc.ipynb" for an example).

In [4]:
from adjacency_graph import Graph

def dfs_visit_collect(self, start, component):
    """DFS visit that records each finished vertex into a component list."""
    start.color = "gray"
    self._time += 1
    start.discovery_time = self._time
    for next_vertex in start.get_neighbors():
        if next_vertex.color == "white":
            next_vertex.set_previous(start)
            self.dfs_visit_collect(next_vertex, component)
    start.color = "black"
    self._time += 1
    start.closing_time = self._time
    component.append(start.key)

def scc(self):
    # DFS on original graph to get closing times
    for vertex in self:
        vertex.color = "white"
    self._time = 0
    for vertex in self:
        if vertex.color == "white":
            self.dfs_visit(vertex)

    # Build transpose
    gt = self.transpose()

    # Copy closing times from original to transpose vertices
    for key in self._vertices:
        gt.get_vertex(key).closing_time = self._vertices[key].closing_time

    # DFS on transpose in decreasing closing time order
    for vertex in gt:
        vertex.color = "white"
    gt._time = 0

    sorted_verts = sorted(gt, key=lambda v: v.closing_time, reverse=True)

    components = []
    for vertex in sorted_verts:
        if vertex.color == "white":
            component = []
            gt.dfs_visit_collect(vertex, component)
            components.append(component)

    # Build condensed graph — one vertex per SCC
    result = Graph()
    key_to_scc = {}
    for comp in components:
        label = tuple(sorted(comp))
        result.set_vertex(label)
        for key in comp:
            key_to_scc[key] = label

    for (from_key, to_key) in self._edges:
        from_scc = key_to_scc[from_key]
        to_scc = key_to_scc[to_key]
        if from_scc != to_scc:
            result.add_edge(from_scc, to_scc)

    return result, components

Graph.dfs_visit_collect = dfs_visit_collect
Graph.scc = scc

# test
G = Graph()
for u, v in [("A","B"),("B","C"),("C","A"),("B","D"),("D","E"),("E","F"),("F","D"),("G","F")]:
    G.add_edge(u, v)

condensed, components = G.scc()
print("SCCs:")
for comp in components:
    print(" ", sorted(comp))
print("Condensed graph edges:")
for edge in condensed.get_edges():
    print(" ", edge)

SCCs:
  ['G']
  ['A', 'B', 'C']
  ['D', 'E', 'F']
Condensed graph edges:
  (('A', 'B', 'C'), ('D', 'E', 'F'))
  (('G',), ('D', 'E', 'F'))


<font size = "3">

**(Q4)** Write a program to solve the following problem: you have two jugs, a 4-gallon and a 3-gallon. Neither of the jugs has any markings. There is a pump that can be used to fill the jugs with water. How can you get exactly two gallons of water in the 4-gallon jug?


**Hint:** Represent this as a graph, where the key of each vertex is an ordered pair of integers $(a, b)$ satisfying

$$0\leq a\leq 4,\quad 0\leq b\leq 3$$



In [5]:
from adjacency_graph import Graph
import sys

def build_jug_graph():
    cap_large, cap_small = 4, 3
    G = Graph()
    for a in range(cap_large + 1):
        for b in range(cap_small + 1):
            state = (a, b)
            pour_ls = min(a, cap_small - b)
            pour_sl = min(b, cap_large - a)
            neighbors = [
                (cap_large, b),          # fill large
                (a, cap_small),          # fill small
                (0, b),                  # empty large
                (a, 0),                  # empty small
                (a - pour_ls, b + pour_ls),  # pour large into small
                (a + pour_sl, b - pour_sl),  # pour small into large
            ]
            for neighbor in neighbors:
                if neighbor != state:    # skip self-loops
                    G.add_edge(state, neighbor)
    return G

def solve_water_jug():
    G = build_jug_graph()
    start = G.get_vertex((0, 0))
    G.bfs(start)

    goal = min(
        (G.get_vertex(key) for key in G.get_vertices() if key[0] == 2),
        key=lambda v: v.distance
    )

    if goal.distance == sys.maxsize:
        print("No solution found.")
        return

    path = []
    current = goal
    while current:
        path.append(current.key)
        current = current.previous
    path.reverse()

    print("Steps to get exactly 2 gallons in the 4-gallon jug:")
    for state in path:
        print(f"  4-gal jug: {state[0]},  3-gal jug: {state[1]}")

solve_water_jug()

Steps to get exactly 2 gallons in the 4-gallon jug:
  4-gal jug: 0,  3-gal jug: 0
  4-gal jug: 0,  3-gal jug: 3
  4-gal jug: 3,  3-gal jug: 0
  4-gal jug: 3,  3-gal jug: 3
  4-gal jug: 4,  3-gal jug: 2
  4-gal jug: 0,  3-gal jug: 2
  4-gal jug: 2,  3-gal jug: 0


<font size = "3">

**(Q5)** Generalize the problem above so that the parameters to your solution include the size of each jug and the final amount of water to be left in the larger jug.

In [6]:
from adjacency_graph import Graph
import sys

def build_jug_graph_general(cap_large, cap_small):
    G = Graph()
    for a in range(cap_large + 1):
        for b in range(cap_small + 1):
            state = (a, b)
            pour_ls = min(a, cap_small - b)
            pour_sl = min(b, cap_large - a)
            neighbors = [
                (cap_large, b),
                (a, cap_small),
                (0, b),
                (a, 0),
                (a - pour_ls, b + pour_ls),
                (a + pour_sl, b - pour_sl),
            ]
            for neighbor in neighbors:
                if neighbor != state:   
                    G.add_edge(state, neighbor)
    return G

def solve_water_jug_general(cap_large, cap_small, target):
    G = build_jug_graph_general(cap_large, cap_small)
    start = G.get_vertex((0, 0))
    G.bfs(start)

    goal_states = [
        G.get_vertex(key) for key in G.get_vertices()
        if key[0] == target and G.get_vertex(key).distance < sys.maxsize
    ]

    if not goal_states:
        print(f"No solution: cannot get exactly {target} gallons in the {cap_large}-gallon jug.")
        return

    goal = min(goal_states, key=lambda v: v.distance)

    path = []
    current = goal
    while current:
        path.append(current.key)
        current = current.previous
    path.reverse()

    print(f"Solution ({cap_large}-gal jug, {cap_small}-gal jug, target = {target} gal):")
    for state in path:
        print(f"  {cap_large}-gal jug: {state[0]},  {cap_small}-gal jug: {state[1]}")

# Original problem
solve_water_jug_general(4, 3, 2)

print()

# Try a different configuration
solve_water_jug_general(5, 3, 4)

Solution (4-gal jug, 3-gal jug, target = 2 gal):
  4-gal jug: 0,  3-gal jug: 0
  4-gal jug: 0,  3-gal jug: 3
  4-gal jug: 3,  3-gal jug: 0
  4-gal jug: 3,  3-gal jug: 3
  4-gal jug: 4,  3-gal jug: 2
  4-gal jug: 0,  3-gal jug: 2
  4-gal jug: 2,  3-gal jug: 0

Solution (5-gal jug, 3-gal jug, target = 4 gal):
  5-gal jug: 0,  3-gal jug: 0
  5-gal jug: 5,  3-gal jug: 0
  5-gal jug: 2,  3-gal jug: 3
  5-gal jug: 2,  3-gal jug: 0
  5-gal jug: 0,  3-gal jug: 2
  5-gal jug: 5,  3-gal jug: 2
  5-gal jug: 4,  3-gal jug: 3
